In [1]:
# ================================================
# ABLATION STUDY: MSL+MER vs +M2020 (completo y reducido)
# ================================================
# Python 3.9 + PyTorch + CUDA 12.1 (RTX 4050)
# Ejecuta esto después de haber corrido el EDA y tener las rutas correctas

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

# ====================== 1. DEVICE ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device} (RTX 4050 con CUDA 12.1)")


Usando dispositivo: cuda (RTX 4050 con CUDA 12.1)


In [2]:
print("CUDA disponible:", torch.cuda.is_available())
print("Versión CUDA en PyTorch:", torch.version.cuda)
print("Dispositivos:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA disponible: True
Versión CUDA en PyTorch: 12.1
Dispositivos: 1
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
# === OPTIMIZACIONES DE VELOCIDAD (agrega justo después de los imports) ===
import torchvision.transforms as T
torch.backends.cudnn.benchmark = True          # acelera convoluciones
print("✅ Optimizaciones de velocidad activadas")

✅ Optimizaciones de velocidad activadas


In [4]:

# ====================== 2. FUNCIÓN DE PAIRING (la que te dio 25k en MSL) ======================
def build_pairs_fast(image_dir, mask_dir):
    pairs = []
    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg'))]
    mask_files = [m for m in os.listdir(mask_dir) if m.lower().endswith('.png')]
    
    for img_name in image_files:
        base = os.path.splitext(img_name)[0]
        img_path = os.path.join(image_dir, img_name)
        
        for mask_name in mask_files:
            if mask_name.startswith(base):          # clave: startswith (igual que en tu EDA)
                mask_path = os.path.join(mask_dir, mask_name)
                pairs.append((img_path, mask_path))
                break
    return pairs


In [5]:

# ====================== 3. CONSTRUIR PARES (usa tus rutas exactas) ======================
base_path = r"C:\Users\User\Documents\DeepLearning\ai4mars-dataset-merged-0.6"

# MSL (mcam + ncam)
msl_mcam_img = os.path.join(base_path, "msl", "mcam", "images")
msl_mcam_lbl = os.path.join(base_path, "msl", "mcam", "labels", "train")
msl_ncam_img = os.path.join(base_path, "msl", "ncam", "images", "edr")
msl_ncam_lbl = os.path.join(base_path, "msl", "ncam", "labels", "train")

msl_paths = build_pairs_fast(msl_mcam_img, msl_mcam_lbl)
msl_paths += build_pairs_fast(msl_ncam_img, msl_ncam_lbl)

# MER
mer_img = os.path.join(base_path, "mer", "images", "eff")
mer_lbl = os.path.join(base_path, "mer", "labels", "train", "merged-unmasked")
mer_paths = build_pairs_fast(mer_img, mer_lbl)

# M2020 (NAV) - solo para los modelos B y C
nav_img = os.path.join(base_path, "m2020", "images", "ncam")
nav_lbl = os.path.join(base_path, "m2020", "labels", "NAV")
nav_paths = build_pairs_fast(nav_img, nav_lbl)

print(f"MSL pares totales: {len(msl_paths)}")
print(f"MER pares totales: {len(mer_paths)}")
print(f"NAV (M2020) pares totales: {len(nav_paths)}")


MSL pares totales: 25163
MER pares totales: 8303
NAV (M2020) pares totales: 2642


In [6]:

# ====================== 4. SPLIT TRAIN/VAL (solo MSL+MER - val común para los 3 modelos) ======================
random.seed(42)  # reproducibilidad

# 10% para validación (solo MSL+MER)
val_size_msl = int(len(msl_paths) * 0.1)
val_size_mer = int(len(mer_paths) * 0.1)

msl_val = msl_paths[:val_size_msl]
msl_train = msl_paths[val_size_msl:]

mer_val = mer_paths[:val_size_mer]
mer_train = mer_paths[val_size_mer:]

val_pairs = msl_val + mer_val   # ← val común (nunca ve M2020)

print(f"Val set (MSL+MER): {len(val_pairs)} pares")


Val set (MSL+MER): 3346 pares


In [7]:

# ====================== 5. DATASET (actualizado y mejorado) ======================
class MarsBalancedDataset(Dataset):
    def __init__(self, msl_paths, mer_paths, nav_paths=None, length=30000, rover_weights=None):
        self.msl_paths = msl_paths
        self.mer_paths = mer_paths
        self.nav_paths = nav_paths or []
        self.length = length
        
        self.rover_list = ["MSL", "MER"]
        if len(self.nav_paths) > 0:
            self.rover_list.append("NAV")
        
        self.rover_weights = rover_weights or [1.0 / len(self.rover_list)] * len(self.rover_list)

        # Transformaciones rápidas (mucho más rápido que PIL manual)
        self.transform = T.Compose([
            T.Resize((512, 512), interpolation=T.InterpolationMode.BILINEAR),
            T.ToTensor(),                    # ya hace /255.0 y pasa a [0,1]
        ])

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        rover = random.choices(self.rover_list, weights=self.rover_weights, k=1)[0]

        if rover == "MSL":
            img_path, mask_path = random.choice(self.msl_paths)
        elif rover == "MER":
            img_path, mask_path = random.choice(self.mer_paths)
        else:
            img_path, mask_path = random.choice(self.nav_paths)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        image = self.transform(image)                    # ← mucho más rápido
        mask = mask.resize((512, 512), Image.NEAREST)   # solo máscara sigue con NEAREST
        mask = torch.from_numpy(np.array(mask)).long()

        mask = torch.clamp(mask, 0, 3)   # ← sigue teniendo la protección

        return image, mask



In [8]:

# ====================== 6. VAL DATASET (determinístico) ======================
class ValDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        image = image.resize((512, 512), Image.BILINEAR)
        mask = mask.resize((512, 512), Image.NEAREST)

        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(np.array(mask)).long()
        return image, mask



In [9]:

# ====================== 7. MODELO: UNet ligera (sin instalaciones extra) ======================
class SimpleUNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 2, stride=2), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 2, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, num_classes, 1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


In [10]:

# ====================== 8. MÉTRICA mIoU (CORREGIDA) ======================
def compute_miou(pred, target, num_classes=4, ignore_index=255):
    pred = pred.argmax(dim=1)
    
    # === FIX CRÍTICO: mismo dispositivo ===
    target = target.to(pred.device)
    
    mask = (target != ignore_index)
    pred = pred[mask]
    target = target[mask]
    
    iou_per_class = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()
        if union == 0:
            iou_per_class.append(torch.tensor(1.0, device=pred.device))
        else:
            iou_per_class.append(intersection / union)
    
    return torch.mean(torch.stack(iou_per_class)).item()



In [16]:

# ====================== 9. FUNCIÓN DE ENTRENAMIENTO (versión Windows-friendly) ======================
def train_model(train_dataset, val_pairs, model_name="Model", epochs=5, batch_size=16, lr=1e-3):
    # Batch más grande + AMP
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=True, persistent_workers=False)
    val_dataset = ValDataset(val_pairs)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=True)
    model = SimpleUNet(num_classes=4).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=255)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # === MIXED PRECISION (lo que más acelera en RTX 4050) ===
    scaler = torch.amp.GradScaler(device = 'cuda')

    best_miou = 0.0
    print(f"\n=== Entrenando {model_name} (batch={batch_size} + AMP) ===")

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.amp.autocast(device_type = 'cuda'):          # ← AMP
                outputs = model(images)
                loss = criterion(outputs, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

        # Validación (igual que antes)
        model.eval()
        val_miou = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                outputs = model(images)
                val_miou += compute_miou(outputs, masks)

        val_miou /= len(val_loader)
        print(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f} | Val mIoU: {val_miou:.4f}")

        if val_miou > best_miou:
            best_miou = val_miou

    print(f"FINAL {model_name} → Best Val mIoU: {best_miou:.4f}")
    return best_miou



In [12]:

# ====================== 10. CREAR LOS 3 DATASETS PARA ABLATION ======================
# Modelo A: SOLO MSL + MER (baseline)
dataset_A = MarsBalancedDataset(msl_train, mer_train, None, length=30000, rover_weights=[0.5, 0.5])

# Modelo B: MSL + MER + M2020 completo (igual probabilidad)
dataset_B = MarsBalancedDataset(msl_train, mer_train, nav_paths, length=30000)

# Modelo C: MSL + MER + M2020 reducido a 20k imágenes + menor peso a M2020 (para disminuir ruido de dominio)
dataset_C = MarsBalancedDataset(msl_train, mer_train, nav_paths, length=20000, rover_weights=[0.45, 0.45, 0.10])


In [13]:
# === PRUEBA RÁPIDA DE QUE EL DATASET FUNCIONA ===
print("Probando carga de un batch (debe tardar < 5 segundos)...")
test_loader = DataLoader(dataset_A, batch_size=4, shuffle=False, num_workers=0, pin_memory=True)
for images, masks in test_loader:
    print(f"✅ Batch cargado correctamente! Shape imagen: {images.shape}, máscara: {masks.shape}")
    print(f"GPU memory usada: {torch.cuda.memory_allocated()/1024**2:.1f} MB")
    break

Probando carga de un batch (debe tardar < 5 segundos)...
✅ Batch cargado correctamente! Shape imagen: torch.Size([4, 3, 512, 512]), máscara: torch.Size([4, 512, 512])
GPU memory usada: 0.0 MB


In [17]:

# ====================== 11. EJECUTAR LOS 3 MODELOS ======================
miou_A = train_model(dataset_A, val_pairs, model_name="A (MSL+MER)", epochs=3, batch_size=16)
miou_B = train_model(dataset_B, val_pairs, model_name="B (MSL+MER+NAV full)", epochs=3, batch_size=16)
miou_C = train_model(dataset_C, val_pairs, model_name="C (20k)", epochs=3, batch_size=16)

print("\n" + "="*60)
print("RESULTADOS ABLATION STUDY")
print("="*60)
print(f"Modelo A (solo MSL+MER)          → mIoU = {miou_A:.4f}")
print(f"Modelo B (+M2020 completo)        → mIoU = {miou_B:.4f}")
print(f"Modelo C (+M2020 reducido a 20k)  → mIoU = {miou_C:.4f}")
print("="*60)


=== Entrenando A (MSL+MER) (batch=16 + AMP) ===


Epoch 1/3: 100%|██████████| 1875/1875 [58:11<00:00,  1.86s/it]


Epoch 1 - Train Loss: 1.1687 | Val mIoU: 0.2472


Epoch 2/3:  49%|████▉     | 915/1875 [33:02<34:40,  2.17s/it]  


KeyboardInterrupt: 